# DistilBERT Fake Review Detection + ONNX Export
**Model:** `distilbert-base-uncased` (fine-tuned for fake review detection)

**Dataset:** `fake reviews dataset.csv` (Kaggle)

**Labels:** CG (Computer Generated) = 1 (fake) | OR (Original) = 0 (genuine)

**Training sample:** All 40,432 rows (20,216 fake + 20,216 genuine) — full dataset.

---
### Instructions
1. Go to **Runtime → Change runtime type → A100 GPU** before running.
2. Upload `fake reviews dataset.csv` to your Google Drive.
3. Run all cells in order.
4. Download `fake_distilbert_onnx.zip` and extract it into your `data/` folder.

In [ ]:
import torch
print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))

In [ ]:
!pip install -q transformers accelerate optimum[onnxruntime]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── UPDATE THIS PATH if the file is inside a folder ──
CSV_PATH = '/content/drive/MyDrive/fake reviews dataset.csv'

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────
MAX_LEN       = 128
BATCH_SIZE    = 32
EPOCHS        = 3
LEARNING_RATE = 2e-5
MODEL_NAME    = 'distilbert-base-uncased'
# ─────────────────────────────────────────────────────────────────

In [ ]:
# Load fake reviews dataset
print('Loading data ...')
df = pd.read_csv(CSV_PATH)
print(f'Total rows: {len(df):,}')
print(df['label'].value_counts())

# CG = 1 (fake), OR = 0 (genuine)
df['label_int'] = df['label'].map({'CG': 1, 'OR': 0})
df = df.dropna(subset=['text_', 'label_int'])
df['text_'] = df['text_'].astype(str)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'\nUsing {len(df):,} rows  ({df["label_int"].sum():,} fake / {(df["label_int"]==0).sum():,} genuine)')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text_'].tolist(),
    df['label_int'].tolist(),
    test_size=0.2,
    random_state=SEED,
    stratify=df['label_int'].tolist()
)
print(f'Train: {len(X_train):,}   Test: {len(X_test):,}')

In [ ]:
print(f'Loading tokenizer: {MODEL_NAME} ...')
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
print('Done.')

In [ ]:
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(
            texts, truncation=True, padding='max_length',
            max_length=max_len, return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx],
        }

print('Tokenising training set ...')
train_dataset = ReviewDataset(X_train, y_train, tokenizer, MAX_LEN)
print('Tokenising test set ...')
test_dataset  = ReviewDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)
print(f'Train batches: {len(train_loader)}   Test batches: {len(test_loader)}')

In [ ]:
print(f'Loading model: {MODEL_NAME} ...')
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = model.to(DEVICE)
print('Model loaded.')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
optimizer    = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = total_steps // 10
scheduler    = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
print(f'Total steps: {total_steps:,}   Warmup: {warmup_steps:,}')

In [ ]:
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch['labels'].numpy())
    return accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds)

best_f1       = 0
train_history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += outputs.loss.item()
        if (step + 1) % 100 == 0:
            print(f'  Epoch {epoch} | Step {step+1}/{len(train_loader)} | Loss: {total_loss/(step+1):.4f}')

    avg_loss        = total_loss / len(train_loader)
    val_acc, val_f1 = evaluate(model, test_loader)
    train_history.append({'epoch': epoch, 'loss': avg_loss, 'val_acc': val_acc, 'val_f1': val_f1})
    print(f'\nEpoch {epoch}/{EPOCHS} — Loss: {avg_loss:.4f} | Val Accuracy: {val_acc:.1%} | Val F1: {val_f1:.1%}')

    if val_f1 > best_f1:
        best_f1 = val_f1
        model.save_pretrained('fake_distilbert_model')
        tokenizer.save_pretrained('fake_distilbert_model')
        print(f'  ✓ Best model saved (F1 = {best_f1:.1%})')
    print()

In [ ]:
# Final evaluation
best_model = DistilBertForSequenceClassification.from_pretrained('fake_distilbert_model').to(DEVICE)
best_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        outputs = best_model(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE)
        )
        all_preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
        all_labels.extend(batch['labels'].numpy())

print('=== Final Test Results ===')
print(classification_report(all_labels, all_preds, target_names=['Genuine', 'Fake']))

In [ ]:
# Export to ONNX
print('Exporting to ONNX ...')
from optimum.onnxruntime import ORTModelForSequenceClassification

ort_model = ORTModelForSequenceClassification.from_pretrained('fake_distilbert_model', export=True)
ort_model.save_pretrained('fake_distilbert_onnx')
tokenizer.save_pretrained('fake_distilbert_onnx')
print('ONNX model saved to: fake_distilbert_onnx/')

In [ ]:
# Verify ONNX model
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import pipeline

ort_model = ORTModelForSequenceClassification.from_pretrained(
    'fake_distilbert_onnx', provider='CPUExecutionProvider'
)
pipe = pipeline('text-classification', model=ort_model, tokenizer=tokenizer, device=-1)

samples = [
    'This product is absolutely amazing! Best purchase ever. Highly recommend to everyone!',
    'Broke after two days. Very disappointed with the quality. Would not buy again.',
    'Decent product, does what it says. Nothing special.',
]
for text, result in zip(samples, pipe(samples)):
    label = 'FAKE' if result['label'] == 'LABEL_1' else 'GENUINE'
    print(f'{label} ({result["score"]:.1%}) — "{text[:60]}"')

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

epochs_range = [h['epoch'] for h in train_history]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs_range, [h['loss']    for h in train_history], marker='o'); ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch')
ax2.plot(epochs_range, [h['val_acc'] for h in train_history], marker='o', label='Accuracy')
ax2.plot(epochs_range, [h['val_f1']  for h in train_history], marker='s', label='F1')
ax2.set_title('Validation Metrics'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Zip and download
import shutil
shutil.make_archive('fake_distilbert_onnx', 'zip', 'fake_distilbert_onnx')
print('Zipped: fake_distilbert_onnx.zip')

from google.colab import files
files.download('fake_distilbert_onnx.zip')

## Next steps
1. Extract `fake_distilbert_onnx.zip`
2. Place the extracted folder in your `data/` directory:
```
data/fake_distilbert_onnx/
    model.onnx
    config.json
    tokenizer.json
    vocab.txt
    ...
```